<a href="https://colab.research.google.com/github/yeriverie/AI-Pioneers-ML-Internship/blob/main/task4_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ==========================================
# STEP 1: BUILD & TRAIN END-TO-END PIPELINE
# ==========================================
print("--- 1. Training ML Pipeline ---")
np.random.seed(42)
n_samples = 500

# Feature generation
df = pd.DataFrame({
    'Age': np.random.randint(18, 65, n_samples),
    'Salary': np.random.randint(25000, 150000, n_samples),
    'Credit_Score': np.random.randint(300, 850, n_samples)
})

# Target variable: Loan Approval Status (1 = Approved, 0 = Rejected)
df['Approved'] = ((df['Salary'] > 60000) & (df['Credit_Score'] > 650)).astype(int)

X = df[['Age', 'Salary', 'Credit_Score']]
y = df['Approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a scikit-learn Pipeline combining Scaling and Classification
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train pipeline
pipeline.fit(X_train, y_train)

# Save serialized model to disk
joblib.dump(pipeline, 'model.pkl')
print("Model pipeline successfully saved to 'model.pkl'")

# ==========================================
# STEP 2: FLASK API DEPLOYMENT CODE (app.py)
# ==========================================
flask_app_code = """
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

# Load trained pipeline
model = joblib.load('model.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    features = np.array([[data['Age'], data['Salary'], data['Credit_Score']]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]

    return jsonify({
        'prediction': int(prediction),
        'approval_status': 'Approved' if prediction == 1 else 'Rejected',
        'approval_probability': float(probability)
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)
"""

with open('app.py', 'w') as f:
    f.write(flask_app_code)

print("Flask application script written to 'app.py'")

--- 1. Training ML Pipeline ---
Model pipeline successfully saved to 'model.pkl'
Flask application script written to 'app.py'
